# Section 5: High NA Annular Aperture

**System:** NA = 0.9, λ = 532 nm, annular aperture, medium: air (n = 1).

This final notebook combines the two major effects from Sections 3 and 4:
- **Annular aperture** (Babinet's principle): narrower focal spot and extended depth of field
- **High NA vectorial effects**: significant Ez (longitudinal) and Ey (cross-polarized) components

At high NA, the annular aperture strongly emphasizes peripheral rays converging at steep angles, further enhancing vectorial effects. The Ez component can become the dominant contributor to the central intensity for large ε.

**Goals:**
- Focal-plane 1D radial intensity for ε = 0.5 and ε = 0.99 with uniform and Gaussian inputs
- Extended depth of field with annular aperture at high NA
- Vectorial field components (Ex, Ey, Ez) for high NA annular aperture
- Use `annular_intensity()` and `annular_field()` from the package

> **Computation note:** Each `annular_intensity()` call runs two full RW integrations (outer + inner disk). With 70 radial points this takes a few minutes. Keep arrays small.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import sys
sys.path.insert(0, '/Users/raaromero/Projects/Research/optical-diffraction/src')

from optical_diffraction import annular_field, annular_intensity

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

# System parameters
NA = 0.9
wavelength = 0.532   # microns
n_medium = 1.0

airy_radius = 0.61 * wavelength / NA
dof = wavelength / NA**2

print(f"System: NA={NA}, λ={wavelength} μm")
print(f"Paraxial Airy radius: {airy_radius:.3f} μm")
print(f"Paraxial DoF (λ/NA²): {dof:.3f} μm")
print(f"\nNote: Annular apertures will extend the depth of field significantly.")

## 5.1 Focal Plane Radial Intensity — ε = 0.5

Uniform and Gaussian inputs through an annular aperture with ε = 0.5 at high NA.

In [ ]:
# Radial grid
r_max = 4.5 * airy_radius
r = np.linspace(0, r_max, 60)
z_focal = np.zeros_like(r)

epsilon_05 = 0.5
epsilon_099 = 0.99

# Input field configurations
configs = [
    ('uniform', 0.0, 'Uniform', 'tab:blue'),
    ('gaussian', 1.0, 'Gaussian α=1', 'tab:orange'),
    ('gaussian', 2.0, 'Gaussian α=2', 'tab:green'),
    ('gaussian', 4.0, 'Gaussian α=4', 'tab:red'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for eps_val, ax in zip([0.0, epsilon_05], axes):
    for (field_type, trunc, label, color) in configs:
        I = annular_intensity(
            NA=NA, epsilon=eps_val, r=r, z=z_focal,
            wavelength=wavelength, n_medium=n_medium,
            input_field=field_type, truncation_coeff=trunc,
        )
        ax.plot(r / airy_radius, I, color=color, lw=2, label=label)

    ax.axvline(x=1.0, color='black', ls=':', lw=1.2, alpha=0.5, label='Airy radius')
    ax.set_xlabel('r / r$_{Airy}^{paraxial}$')
    ax.set_ylabel('Normalized intensity')
    ax.set_xlim(0, r_max / airy_radius)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)

axes[0].set_title(f'Full Disk ε=0 — Reference')
axes[1].set_title(f'Annular ε={epsilon_05}')

fig.suptitle(f'Focal Plane Radial Intensity — High NA (NA={NA}), Annular ε={epsilon_05}', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5.2 Focal Plane Radial Intensity — ε = 0.99

Near-ring aperture at high NA — strong annular effect combined with vectorial diffraction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for eps_val, ax in zip([0.0, epsilon_099], axes):
    for (field_type, trunc, label, color) in configs:
        I = annular_intensity(
            NA=NA, epsilon=eps_val, r=r, z=z_focal,
            wavelength=wavelength, n_medium=n_medium,
            input_field=field_type, truncation_coeff=trunc,
        )
        ax.plot(r / airy_radius, I, color=color, lw=2, label=label)

    ax.axvline(x=1.0, color='black', ls=':', lw=1.2, alpha=0.5, label='Airy radius')
    ax.set_xlabel('r / r$_{Airy}^{paraxial}$')
    ax.set_ylabel('Normalized intensity')
    ax.set_xlim(0, r_max / airy_radius)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)

axes[0].set_title(f'Full Disk ε=0 — Reference')
axes[1].set_title(f'Annular ε={epsilon_099}')

fig.suptitle(f'Focal Plane Radial Intensity — High NA (NA={NA}), Annular ε={epsilon_099}', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5.3 Axial Intensity — Extended Depth of Field at High NA

Comparing on-axis intensity I(r=0, z) for ε = 0, 0.5, 0.99 at high NA.

In [ ]:
# Axial grid
z_max = 5.0 * dof
z_axial = np.linspace(-z_max, z_max, 50)
r_zero = np.zeros_like(z_axial)

epsilons = [0.0, 0.5, 0.99]
eps_colors = ['tab:blue', 'tab:orange', 'tab:red']
eps_styles = ['-', '--', ':']
eps_labels = ['ε=0 (full disk)', 'ε=0.5', 'ε=0.99']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Uniform illumination axial profiles
ax = axes[0]
ax.set_title('Axial Intensity — Uniform Input')
for eps, color, ls, label in zip(epsilons, eps_colors, eps_styles, eps_labels):
    I_ax = annular_intensity(
        NA=NA, epsilon=eps, r=r_zero, z=z_axial,
        wavelength=wavelength, n_medium=n_medium,
        input_field='uniform', truncation_coeff=0.0,
    )
    ax.plot(z_axial / dof, I_ax, color=color, lw=2.5, ls=ls, label=label)

# Gaussian α=2 axial profiles
ax = axes[1]
ax.set_title('Axial Intensity — Gaussian Input (α=2)')
for eps, color, ls, label in zip(epsilons, eps_colors, eps_styles, eps_labels):
    I_ax = annular_intensity(
        NA=NA, epsilon=eps, r=r_zero, z=z_axial,
        wavelength=wavelength, n_medium=n_medium,
        input_field='gaussian', truncation_coeff=2.0,
    )
    ax.plot(z_axial / dof, I_ax, color=color, lw=2.5, ls=ls, label=label)

for ax in axes:
    ax.axvline(x=0, color='black', ls='-', lw=0.8, alpha=0.4)
    ax.axhline(y=0.5, color='gray', ls=':', lw=1, alpha=0.6, label='50% level')
    ax.set_xlabel('z / DoF  (DoF = λ/NA²)')
    ax.set_ylabel('Normalized on-axis intensity I(r=0, z)')
    ax.legend(fontsize=9)

fig.suptitle(f'Axial Intensity — Extended Depth of Field (High NA={NA})', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f"DoF reference: λ/NA² = {dof:.3f} μm at NA={NA}")
print("Larger ε extends the depth of field even at high NA.")

## 5.4 Vectorial Field Components — Annular Aperture at High NA

Using `annular_field()` to directly access Ex, Ey, Ez for the annular aperture. At high NA and large ε, peripheral (high-angle) rays dominate, strongly enhancing Ez.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

eps_cases = [0.5, 0.99]
row_labels = [f'ε={eps}' for eps in eps_cases]
comp_titles = ['|Ex|² (dominant)', '|Ey|² (cross-pol)', '|Ez|² (longitudinal)']

for row, eps_val in enumerate(eps_cases):
    # Uniform input to isolate geometric (aperture) effects
    Ex, Ey, Ez = annular_field(
        NA=NA, epsilon=eps_val, r=r, z=z_focal,
        wavelength=wavelength, n_medium=n_medium,
        input_field='uniform', truncation_coeff=0.0,
    )

    Ix = np.abs(Ex)**2
    Iy = np.abs(Ey)**2
    Iz = np.abs(Ez)**2
    I_tot = Ix + Iy + Iz
    scale = I_tot.max() if I_tot.max() > 0 else 1.0

    for col, (I_comp, comp_label) in enumerate(zip([Ix, Iy, Iz], comp_titles)):
        ax = axes[row, col]
        ax.plot(r / airy_radius, I_comp / scale, lw=2.5,
                color=['tab:blue', 'tab:orange', 'tab:red'][col], label=comp_label)
        ax.plot(r / airy_radius, I_tot / scale, lw=1.5, color='black',
                ls='--', alpha=0.4, label='Total')
        ax.axvline(x=1.0, color='gray', ls=':', lw=1.0)
        ax.set_xlabel('r / r$_{Airy}$')
        ax.set_ylabel('Intensity / I$_{max}$')
        ax.set_title(f'{row_labels[row]}: {comp_label}')
        ax.set_xlim(0, r_max / airy_radius)
        ax.legend(fontsize=9)

    # Print peak ratios
    print(f"ε={eps_val} (uniform): |Ex|²_max={Ix.max()/scale:.4f}, "
          f"|Ey|²_max={Iy.max()/scale:.4f}, |Ez|²_max={Iz.max()/scale:.4f}")

fig.suptitle(f'Vectorial Field Components — High NA (NA={NA}), Annular Aperture, Uniform Input',
             fontsize=13)
plt.tight_layout()
plt.show()

## 5.5 Vectorial Components — Gaussian Inputs Through Annular Apertures

Showing how the Ez fraction changes with input field profile for both ε values.

In [ ]:
alphas_plot = [1, 2, 4]
colors_alpha = ['tab:orange', 'tab:green', 'tab:red']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Rows: ε=0.5, ε=0.99. Columns: total intensity, Ez component
for row, eps_val in enumerate([0.5, 0.99]):
    ax_tot = axes[row, 0]
    ax_ez  = axes[row, 1]

    # Uniform baseline
    Ex_u, Ey_u, Ez_u = annular_field(
        NA=NA, epsilon=eps_val, r=r, z=z_focal,
        wavelength=wavelength, n_medium=n_medium,
        input_field='uniform', truncation_coeff=0.0,
    )
    I_u = np.abs(Ex_u)**2 + np.abs(Ey_u)**2 + np.abs(Ez_u)**2
    peak_u = I_u.max() if I_u.max() > 0 else 1.0
    ax_tot.plot(r / airy_radius, I_u / peak_u, color='tab:blue', lw=2, label='Uniform')
    ax_ez.plot(r / airy_radius, np.abs(Ez_u)**2 / peak_u, color='tab:blue', lw=2, label='Uniform')

    for alpha, color in zip(alphas_plot, colors_alpha):
        Ex_g, Ey_g, Ez_g = annular_field(
            NA=NA, epsilon=eps_val, r=r, z=z_focal,
            wavelength=wavelength, n_medium=n_medium,
            input_field='gaussian', truncation_coeff=float(alpha),
        )
        I_g = np.abs(Ex_g)**2 + np.abs(Ey_g)**2 + np.abs(Ez_g)**2
        peak_g = I_g.max() if I_g.max() > 0 else 1.0
        ax_tot.plot(r / airy_radius, I_g / peak_g, color=color, lw=2, label=f'Gaussian α={alpha}')
        ax_ez.plot(r / airy_radius, np.abs(Ez_g)**2 / peak_g, color=color, lw=2, label=f'α={alpha}')

    for ax in [ax_tot, ax_ez]:
        ax.axvline(x=1.0, color='black', ls=':', lw=1.0, alpha=0.5)
        ax.set_xlabel('r / r$_{Airy}$')
        ax.set_ylabel('Normalized intensity')
        ax.set_xlim(0, r_max / airy_radius)
        ax.legend(fontsize=9)

    ax_tot.set_title(f'ε={eps_val}: Total Intensity')
    ax_ez.set_title(f'ε={eps_val}: |Ez|² Component (Longitudinal)')

fig.suptitle(f'High NA (NA={NA}) Annular Aperture — Total Intensity and Ez Component',
             fontsize=13)
plt.tight_layout()
plt.show()

## 5.6 Full Summary: All ε Values × All Gaussian Inputs — Focal Plane

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes_flat = axes.flatten()

field_cases = [
    ('uniform', 0.0, 'Uniform'),
    ('gaussian', 1.0, 'Gaussian α=1'),
    ('gaussian', 2.0, 'Gaussian α=2'),
    ('gaussian', 4.0, 'Gaussian α=4'),
]

eps_all = [0.0, 0.5, 0.99]
eps_colors_all = ['tab:blue', 'tab:orange', 'tab:red']
eps_styles_all = ['-', '--', ':']
eps_labels_all = ['ε=0 (full disk)', 'ε=0.5', 'ε=0.99']

for ax, (field_type, trunc, field_label) in zip(axes_flat, field_cases):
    for eps, color, ls, label in zip(eps_all, eps_colors_all, eps_styles_all, eps_labels_all):
        I = annular_intensity(
            NA=NA, epsilon=eps, r=r, z=z_focal,
            wavelength=wavelength, n_medium=n_medium,
            input_field=field_type, truncation_coeff=trunc,
        )
        ax.plot(r / airy_radius, I, color=color, lw=2, ls=ls, label=label)

    ax.axvline(x=1.0, color='black', ls=':', lw=1.0, alpha=0.4)
    ax.set_xlabel('r / r$_{Airy}$')
    ax.set_ylabel('Normalized intensity')
    ax.set_title(f'Input: {field_label}')
    ax.set_xlim(0, r_max / airy_radius)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)

fig.suptitle(f'Focal Plane: All Apertures × All Inputs — High NA (NA={NA}, λ={wavelength} μm)',
             fontsize=13)
plt.tight_layout()
plt.show()

## Summary

| Configuration | Central lobe | Side lobes | DoF | Ez fraction |
|---|---|---|---|---|
| ε=0, uniform | Airy-like, vectorially modified | Low | ~λ/NA² | ~10–15% |
| ε=0.5, uniform | Narrower | Moderate | Extended | Higher |
| ε=0.99, uniform | Very narrow | Strong | Most extended | Highest |
| Any ε, Gaussian α=4 | Broadened vs uniform | Lower side lobes | Similar | Similar |

**Key findings for high NA annular apertures:**

1. **Vectorial effects are amplified**: Annular apertures preferentially pass high-angle (peripheral) rays, which carry more longitudinal field. The Ez component grows with ε.
2. **Tighter focus with annular + high NA**: The combination yields the smallest achievable spot for a given wavelength.
3. **Extended depth of field**: Even at high NA, the annular aperture extends the axial intensity profile, useful for confocal and light-sheet microscopy.
4. **Gaussian inputs reduce side lobes** at all ε values, providing a softer annular profile that balances peak intensity with reduced side-lobe contamination.

This completes the five-section analysis of optical diffraction from Gaussian and annular apertures at low and high NA.